# Module 02 — Lecture 2: Shared Memory & Tiling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/GPU-Programming-For-Computational-Neuroscience/blob/main/module_02_memory_optimization/02_shared_memory_tiling.ipynb)

---

Shared memory is the most powerful optimization tool in CUDA. When used correctly, it turns a memory-bandwidth-limited kernel into a compute-limited one — often yielding 5–20× speedups.

**Learning objectives:**
- Implement the tiled matrix multiply algorithm using shared memory
- Explain why tiling improves performance (arithmetic intensity)
- Diagnose and fix shared memory bank conflicts
- Apply tiling to the synaptic input summation problem

In [ ]:
!nvidia-smi

## 1. The Problem: Naive Matrix Multiply is Memory-Bound

Synaptic input to neuron $j$ is: $I_j = \sum_{i} W_{ij} \cdot s_i$

For N neurons this is a matrix-vector product: **I = W · s**.

For a large batched case (computing many I vectors at once): **C = A · B** (matrix multiply).

### Naive GPU implementation

```cpp
__global__ void matmul_naive(float* A, float* B, float* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    float sum = 0;
    for (int k = 0; k < N; k++)
        sum += A[row*N+k] * B[k*N+col];  // ← 2 global reads per iteration
    C[row*N+col] = sum;
}
```

**Arithmetic intensity** = FLOPs / bytes = (2N) / (2×4N) = 0.25 FLOPs/byte

With peak bandwidth of 300 GB/s, this gives: 0.25 × 300 = **75 GFLOPS theoretical max**.

But the GPU can do **10,000 GFLOPS** if we feed it fast enough!

The problem: each element of A is read **N times** (once per output row) from slow global memory.

## 2. Tiling: The Solution

Instead of reading global memory N times per element, load a **tile** (TILE×TILE submatrix) into shared memory once, then reuse it TILE times.

```
Global A (N×N)           Shared tile of A
┌─────────────┐           ┌──────┐
│ . . . . . . │           │tile 0│  loaded once → used TILE_SIZE times
│ . ┌────┐ . │  load ──▶ └──────┘
│ . │tile│ . │           ┌──────┐
│ . └────┘ . │           │tile 1│  next tile
│ . . . . . . │           └──────┘
└─────────────┘
```

**New arithmetic intensity:** 2N FLOPs / (2 × 4 × N/TILE) bytes = TILE × 0.25

With TILE_SIZE=16: intensity = 4 FLOPs/byte → 4 × 300 = **1200 GFLOPS theoretical max**

A 16× improvement in theoretical throughput just from memory access pattern!

In [ ]:
%%writefile matmul_compare.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define TILE 16

__global__ void matmul_naive(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row >= N || col >= N) return;
    float s = 0;
    for (int k = 0; k < N; k++) s += A[row*N+k] * B[k*N+col];
    C[row*N+col] = s;
}

__global__ void matmul_tiled(const float* A, const float* B, float* C, int N) {
    __shared__ float sA[TILE][TILE];
    __shared__ float sB[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    float s = 0;

    for (int t = 0; t < (N + TILE - 1) / TILE; t++) {
        // Load tile collaboratively
        int kA = t * TILE + threadIdx.x;
        int kB = t * TILE + threadIdx.y;
        sA[threadIdx.y][threadIdx.x] = (row<N && kA<N) ? A[row*N+kA] : 0;
        sB[threadIdx.y][threadIdx.x] = (kB<N && col<N) ? B[kB*N+col] : 0;
        __syncthreads();
        // Compute from shared memory
        for (int k = 0; k < TILE; k++) s += sA[threadIdx.y][k] * sB[k][threadIdx.x];
        __syncthreads();
    }
    if (row<N && col<N) C[row*N+col] = s;
}

int main() {
    int sizes[] = {128, 256, 512, 1024};
    printf("%-6s  %-12s  %-12s  %-8s\n", "N", "Naive (ms)", "Tiled (ms)", "Speedup");
    printf("%-6s  %-12s  %-12s  %-8s\n", "------", "----------", "----------", "-------");

    for (int N : sizes) {
        size_t bytes = (size_t)N*N*sizeof(float);
        float *dA, *dB, *dC;
        cudaMalloc(&dA, bytes); cudaMalloc(&dB, bytes); cudaMalloc(&dC, bytes);

        // Random init
        float* hA = (float*)malloc(bytes);
        for (int i = 0; i < N*N; i++) hA[i] = (float)rand()/RAND_MAX;
        cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice);
        cudaMemcpy(dB, hA, bytes, cudaMemcpyHostToDevice);  // same for simplicity
        free(hA);

        dim3 block(TILE, TILE);
        dim3 grid((N+TILE-1)/TILE, (N+TILE-1)/TILE);

        cudaEvent_t t0, t1;
        cudaEventCreate(&t0); cudaEventCreate(&t1);
        float ms_n, ms_t;

        cudaEventRecord(t0);
        matmul_naive<<<grid, block>>>(dA, dB, dC, N);
        cudaEventRecord(t1); cudaEventSynchronize(t1);
        cudaEventElapsedTime(&ms_n, t0, t1);

        cudaEventRecord(t0);
        matmul_tiled<<<grid, block>>>(dA, dB, dC, N);
        cudaEventRecord(t1); cudaEventSynchronize(t1);
        cudaEventElapsedTime(&ms_t, t0, t1);

        printf("%-6d  %-12.2f  %-12.2f  %-8.2f\n", N, ms_n, ms_t, ms_n/ms_t);

        cudaEventDestroy(t0); cudaEventDestroy(t1);
        cudaFree(dA); cudaFree(dB); cudaFree(dC);
    }
    return 0;
}

In [ ]:
!nvcc -O2 -std=c++11 -o matmul_compare matmul_compare.cu && ./matmul_compare

## 3. Bank Conflicts in Shared Memory

Shared memory is divided into **32 banks** (matching warp size). Each bank can serve one request per cycle. If multiple threads in a warp access different addresses in the **same bank**, they are serialized — a **bank conflict**.

```
Shared memory banks:   Bank0  Bank1  Bank2 ... Bank31
  float smem[32]:      smem[0] smem[1] smem[2] ... smem[31]  (one per bank)
  float smem[64]:      smem[0] smem[1] ...    smem[31] | smem[32] smem[33] ...
                        Bank0   Bank1  ...    Bank31   | Bank0   Bank1   ...

CONFLICT-FREE:
  Thread i accesses smem[i]       → all different banks → 1 cycle

2-WAY BANK CONFLICT:
  Thread i accesses smem[2*i]     → threads 0,16 both hit Bank0 → 2 cycles

32-WAY BANK CONFLICT (worst):
  All threads access smem[0]      → all hit Bank0 → 32 cycles
  (Exception: broadcast — all reading the SAME address is conflict-free)
```

### Common Fix: Padding

```cpp
// Tiled matmul: accessing sA[k][threadIdx.x] across k can cause conflicts
__shared__ float sA[TILE][TILE + 1];  // pad by 1 → shifts bank assignments
```

Adding one extra column shifts each row's starting bank, eliminating column-wise conflicts.

## 4. Application: Fast Synaptic Input Summation

The most common operation in a neural network simulation:

$$I_j = \sum_{i=0}^{N-1} W_{ij} \cdot s_i$$

This is a matrix-vector product: **I = W · s** (W is N×N, s and I are N-vectors).

For large N (10,000+), tiling this operation gives a significant speedup over a naive implementation.

In Module 05 we will implement this with sparse connectivity (most $W_{ij} = 0$), which is even more efficient.

## Summary

| Technique | Benefit | When to Use |
|-----------|---------|-------------|
| Shared memory | 30× faster than global | Data reused multiple times within a block |
| Tiling | Increases arithmetic intensity | Matrix ops, convolutions, correlation |
| Padding | Eliminates bank conflicts | Column-wise shared memory access |
| `__syncthreads()` | Correctness barrier | Always after loading shared memory |

**Proceed to:** [Lecture 3: Profiling & Optimization Workflow](03_optimization_workflow.ipynb)